# 🔢 NumPy — Session 1

**Track:** Intro to Machine Learning — Data Analysis

By the end of this session you will be able to:
- Explain why NumPy arrays exist and how they differ from Python lists
- Create arrays in multiple ways (manual, zeros/ones, ranges, random)
- Understand **broadcasting** — the single most important NumPy concept
- Index, slice, and boolean-mask 1D and 2D arrays
- Tell the difference between a **view** and a **copy**
- Use `axis` to compute row-wise / column-wise statistics
- Manipulate array shape and combine arrays
- Connect a couple of these ideas to what you'll use them for in ML later

> 💡 Tip: run every cell yourself, don't just read. NumPy is a "muscle memory" library.

In [1]:
# importing numpy
import numpy as np

## Why NumPy?

If you want to add two lists element-wise like `[1, 2, 3] + [1, 2, 3] = [2, 4, 6]`,
regular Python lists **won't do it mathematically** — `+` on lists means concatenation,
not addition.

In [2]:
list1 = [1, 2, 3]
addition = list1 + list1  # wrong way — this concatenates, it does NOT add
addition

[1, 2, 3, 1, 2, 3]

In [3]:
list1 = np.array([1, 2, 3])
list2 = np.array([1, 1, 1])
addition = list1 + list2  # right way — element-wise addition
addition

array([2, 3, 4])

### NumPy is also much faster

NumPy arrays are stored in contiguous memory blocks and operations are run in compiled C code,
not the Python interpreter loop. This matters a lot once your datasets get big (which they will
in ML).

In [4]:
import time

big_list = list(range(1_000_000))
big_array = np.arange(1_000_000)

# pure python loop
start = time.time()
result_list = [x * 2 for x in big_list]
print("Python list comprehension:", time.time() - start, "seconds")

# vectorized numpy
start = time.time()
result_array = big_array * 2
print("NumPy vectorized:", time.time() - start, "seconds")

Python list comprehension: 0.16983532905578613 seconds
NumPy vectorized: 0.008888483047485352 seconds


**Takeaway:** same result, but the vectorized NumPy version is typically 10–50x faster. This is why virtually every ML library (pandas, scikit-learn, PyTorch, TensorFlow) is built on top of NumPy arrays.

## Creating arrays

In [5]:
a = np.array([1, 2, 3, 4])
a

array([1, 2, 3, 4])

In [6]:
a = np.zeros((2, 3))
a

array([[0., 0., 0.],
       [0., 0., 0.]])

In [7]:
a = np.ones((2, 2))
a

array([[1., 1.],
       [1., 1.]])

In [8]:
a = np.full((2, 2), 7)  # constant array
a

array([[7, 7],
       [7, 7]])

In [9]:
a = np.identity(n=3)  # identity matrix (always square)
# a = np.eye(3)          # eye() can also make non-square matrices, and offset the diagonal with k=
a

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

**Quick check:** what's the difference between `arange` and `linspace`?

- **`np.arange(start, stop, step)`** — you choose the **step size**, NumPy figures out how many values you get.
- **`np.linspace(start, stop, num)`** — you choose the **number of values**, NumPy figures out the step size for you (evenly spaced, `stop` is included by default).

In [10]:
a = np.arange(5, 25, 5)
a

array([ 5, 10, 15, 20])

In [11]:
a = np.linspace(0, 10, 5)  # step = (stop - start) / (num - 1)
a

array([ 0. ,  2.5,  5. ,  7.5, 10. ])

## Random arrays

- **`np.random.rand(x, y)`** — uniform floats between 0 and 1
- **`np.random.randint(start, stop, size=(x, y))`** — random integers
- **`np.random.uniform(start, stop, size=(x, y))`** — random floats in a custom range

`np.random.seed(n)` makes randomness **reproducible** — everyone who runs the cell with the same
seed gets the same "random" numbers. This matters a lot in ML for reproducible experiments.

In [12]:
np.random.seed(77)
a = np.random.rand(2, 2)
a

array([[0.91910903, 0.6421956 ],
       [0.75371223, 0.13931457]])

In [13]:
a = np.random.randint(1, 11, size=(2, 2))
a

array([[ 1, 10],
       [ 8,  6]], dtype=int32)

In [14]:
a = np.random.uniform(1, 10, size=(2, 2))
a

array([[3.93535843, 5.86961039],
       [3.16211658, 5.90880633]])

## Creating 2D arrays and reading their attributes

⚠️ **Common mistake:** in a Jupyter cell, only the output of the **last line** is displayed.
Use `print()` if you want to see more than one thing.

In [15]:
a_2d = np.array([[1, 2, 3],
                  [4, 5, 6],
                  [1, 2, 3],
                  [4, 5, 6]])
a_2d

array([[1, 2, 3],
       [4, 5, 6],
       [1, 2, 3],
       [4, 5, 6]])

In [16]:
print("ndim :", a_2d.ndim)   # number of dimensions
print("shape:", a_2d.shape)  # (rows, columns)
print("size :", a_2d.size)   # total number of elements
print("dtype:", a_2d.dtype)  # data type of the elements

ndim : 2
shape: (4, 3)
size : 12
dtype: int64


In [17]:
arr = np.array([1, 2, 3])
print("ndim :", arr.ndim)
print("shape:", arr.shape)

ndim : 1
shape: (3,)


In [18]:
a_2d = np.arange(5, 25, 5).reshape(2, 2)  # number of elements must match: (stop-start)/step
a_2d

array([[ 5, 10],
       [15, 20]])

In [19]:
a_2d = np.linspace(0, 1, 10).reshape(2, 5)
a_2d

array([[0.        , 0.11111111, 0.22222222, 0.33333333, 0.44444444],
       [0.55555556, 0.66666667, 0.77777778, 0.88888889, 1.        ]])

*Note: `zeros`, `ones`, `full` etc. already take a shape argument directly, so you don't need `reshape` with them.*

## Arithmetic operations

In [20]:
a = np.array([2, 4, 6, 8])
b = np.array([1, 3, 5, 7])

In [21]:
res = np.add(a, b)  # or a + b
res

array([ 3,  7, 11, 15])

In [22]:
res = np.subtract(a, b)  # or a - b
res

array([1, 1, 1, 1])

In [23]:
res = np.divide(a, b)  # or a / b
res

array([2.        , 1.33333333, 1.2       , 1.14285714])

In [24]:
res = np.multiply(a, b)  # or a * b
res

array([ 2, 12, 30, 56])

In [25]:
res = np.power(b, 2)  # or b ** 2
res

array([ 1,  9, 25, 49])

In [26]:
res = np.power(b, a)
res

array([      1,      81,   15625, 5764801])

In [27]:
x = [25, 9, 36]
res = np.sqrt(x)
res

array([5., 3., 6.])

In [28]:
res = np.dot(a, b)  # sum of element-wise products — this is the core operation behind
res                  # linear regression, neural network layers, cosine similarity, etc.

np.int64(100)

## Broadcasting — the most important NumPy idea

You've already been using broadcasting without naming it! When you do `array + scalar`,
NumPy "stretches" the scalar to match the array's shape. Broadcasting is the general rule
NumPy uses to let arrays of **different shapes** work together in element-wise operations.

**The rule (simplified):** compare shapes from right to left. Two dimensions are compatible if
they're equal, or one of them is 1.

In [29]:
# scalar + array — the scalar is "broadcast" to every element
a = np.array([1, 2, 3])
a + 10

array([11, 12, 13])

In [30]:
# 2D array + 1D array — the 1D array is broadcast across every row
matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])
row = np.array([10, 20, 30])
matrix + row

array([[11, 22, 33],
       [14, 25, 36],
       [17, 28, 39]])

In [31]:
# what happens when shapes are NOT compatible — this is a very common bug source
try:
    bad = np.array([1, 2, 3]) + np.array([1, 2])
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (3,) (2,) 


**Why this matters for ML:** normalizing a whole feature matrix by subtracting column means, or scaling every row by a weight vector, is broadcasting in action — you'll see this constantly in preprocessing.

## Conditional arithmetic

In [32]:
a = np.array([10, 15, 20, 25])
a[a > 15] += 5
a

array([10, 15, 25, 30])

## Comparison operations

In [33]:
a = np.array([1, 2, 3])
b = np.array([1, 1, 1])
a == b

array([ True, False, False])

In [34]:
np.array_equal(a, b)

False

In [35]:
a != b

array([False,  True,  True])

In [36]:
a > b

array([False,  True,  True])

## Accessing items in NumPy arrays

In [37]:
a = np.array([10, 20, 30, 40])

In [38]:
a[1]  # access one item

np.int64(20)

In [39]:
a[1:3]  # slicing: access sequential items

array([20, 30])

In [40]:
a[::2]  # step-slicing: access every other item

array([10, 30])

In [41]:
a[[0, 3, 2]]  # fancy indexing: access non-sequential items by their positions

array([10, 40, 30])

In [42]:
# boolean indexing, example: even elements
x = np.array([1, 2, 3, 4])
bool_mask = x % 2 == 0
x[bool_mask]

array([2, 4])

### ⚠️ Views vs. copies

Basic slicing (`a[1:3]`) returns a **view**, not a copy — it shares memory with the original array.
Changing the slice changes the original! Fancy indexing and boolean masks (`a[[0,3,2]]`, `a[mask]`)
**do** return a copy. This trips up almost everyone at least once — always check when in doubt.

In [43]:
a = np.array([10, 20, 30, 40])
view = a[1:3]        # this is a VIEW
view[0] = 999
print("view:  ", view)
print("original a:", a, "<- changed too!")

view:   [999  30]
original a: [ 10 999  30  40] <- changed too!


In [44]:
a = np.array([10, 20, 30, 40])
copy = a[1:3].copy()  # force an actual copy
copy[0] = 999
print("copy:  ", copy)
print("original a:", a, "<- unchanged")

copy:   [999  30]
original a: [10 20 30 40] <- unchanged


## Accessing 2D arrays

In [45]:
a = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

In [46]:
a[1][1]

np.int64(5)

In [47]:
a[:, -2:]

array([[2, 3],
       [5, 6],
       [8, 9]])

In [48]:
a[:, 0]

array([1, 4, 7])

In [49]:
a[1, 0]

np.int64(4)

In [50]:
a[1][0]  # same as a[1, 0], but a[1, 0] is preferred (faster, more idiomatic)

np.int64(4)

In [51]:
a[:2, 1:]  # slicing rows and columns together

array([[2, 3],
       [5, 6]])

In [52]:
a[0, 1:]

array([2, 3])

## 📝 Exercise 1: "Hot Day Finder"

Tasks:
1. Create a 4×7 matrix of random floats between 20 and 40, representing weekly temperatures for a month (4 weeks × 7 days).
2. Create an array called `weekdays` containing day names in order:
   `['Saturday', 'Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']`
3. Use slicing to extract the last week's temperatures (last row of the matrix).
4. Create a boolean mask identifying days where the temperature exceeded 25°C in the last week.
5. Apply this mask to the `weekdays` array to get the names of hot days.

Try it yourself in the cell below before looking at the solution.

In [53]:
# Your code here





<details>
<summary>💡 Click to see one possible solution</summary>

```python
np.random.seed(1)
temperatures = np.random.uniform(20, 40, size=(4, 7))

weekdays = np.array(['Saturday', 'Sunday', 'Monday', 'Tuesday',
                      'Wednesday', 'Thursday', 'Friday'])

last_week = temperatures[-1]              # step 3: slicing
hot_mask = last_week > 25                 # step 4: boolean mask
hot_days = weekdays[hot_mask]             # step 5: apply mask

print("Last week temps:", last_week)
print("Hot days:", hot_days)
```
</details>

In [54]:
# Solution
np.random.seed(1)
temperatures = np.random.uniform(20, 40, size=(4, 7))

weekdays = np.array(['Saturday', 'Sunday', 'Monday', 'Tuesday',
                      'Wednesday', 'Thursday', 'Friday'])

last_week = temperatures[-1]
hot_mask = last_week > 25
hot_days = weekdays[hot_mask]

print("Last week temps:", last_week)
print("Hot days:", hot_days)

Last week temps: [39.36523151 26.26848356 33.84645231 37.52778305 37.89213327 21.70088423
 20.78109566]
Hot days: ['Saturday' 'Sunday' 'Monday' 'Tuesday' 'Wednesday']


## Array manipulation

In [55]:
a = np.array([1, 2, 3, 4])
b = np.array([[1, 2],
              [3, 4]])

In [56]:
a = np.append(a, 5)  # add a new element at the end
a

array([1, 2, 3, 4, 5])

In [57]:
b = np.append(b, [[5, 6]], axis=0)  # add a new row
b

array([[1, 2],
       [3, 4],
       [5, 6]])

In [58]:
b = np.append(b, [[0], [0], [0]], axis=1)  # add a new column
b

array([[1, 2, 0],
       [3, 4, 0],
       [5, 6, 0]])

In [59]:
a = np.insert(a, 2, 0)  # insert(array, index, value) — insert 0 at position 2
a

array([1, 2, 0, 3, 4, 5])

In [60]:
a = np.delete(a, 2)  # delete the element at position 2
a

array([1, 2, 3, 4, 5])

**`resize` vs `reshape`:** `reshape` will NOT change the amount of data — if the new shape
needs more or fewer elements than you have, it raises an error. `resize` handles this by
repeating or truncating data.

In [61]:
x = np.array([1, 2, 3, 4])
x = np.resize(x, (2, 3))  # reshape would error here (4 elements can't fill a 2x3=6 shape)
x

array([[1, 2, 3],
       [4, 1, 2]])

`flatten()` converts a multi-dimensional array into a 1D array.

In [62]:
a = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
a = a.flatten()
a

array([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [63]:
a = np.array([1, 2])
b = np.array([3, 4])
c = np.concatenate((a, b), axis=0)
c

array([1, 2, 3, 4])

`vstack` and `hstack` are shortcuts for stacking arrays vertically/horizontally — very common
when assembling feature matrices.

In [64]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("vstack (rows):\n", np.vstack((a, b)))
print("hstack (side by side):", np.hstack((a, b)))

vstack (rows):
 [[1 2 3]
 [4 5 6]]
hstack (side by side): [1 2 3 4 5 6]


`np.sort` and `np.unique` are two more everyday tools, especially for exploring data.

In [65]:
a = np.array([5, 1, 4, 2, 3])
print("sorted:", np.sort(a))

b = np.array([1, 2, 2, 3, 3, 3])
print("unique values:", np.unique(b))

sorted: [1 2 3 4 5]
unique values: [1 2 3]


## Aggregation functions

⚠️ Avoid naming variables `min` / `max` — you'll shadow Python's built-in `min()`/`max()` functions
for the rest of the notebook. Use `arr_min`, `arr_max`, etc.

In [66]:
a = np.array([0, 10, 30, 50, 10])

print("mean  :", np.mean(a))
print("median:", np.median(a))
print("std   :", np.std(a))
print("sum   :", np.sum(a))
print("min   :", np.min(a))
print("max   :", np.max(a))
print("argmin (index of min):", np.argmin(a))
print("argmax (index of max):", np.argmax(a))

mean  : 20.0
median: 10.0
std   : 17.88854381999832
sum   : 100
min   : 0
max   : 50
argmin (index of min): 0
argmax (index of max): 3


### The `axis` parameter — critical for real data

So far we've only aggregated the *whole* array into one number. In real datasets (rows =
observations, columns = features), you almost always want a statistic **per row** or **per
column** instead.

- `axis=0` → collapse **rows**, i.e. compute one result **per column**
- `axis=1` → collapse **columns**, i.e. compute one result **per row**

In [67]:
matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])

print("Column-wise sum (axis=0):", matrix.sum(axis=0))
print("Row-wise sum    (axis=1):", matrix.sum(axis=1))
print("Column-wise mean(axis=0):", matrix.mean(axis=0))
print("Row-wise mean   (axis=1):", matrix.mean(axis=1))

Column-wise sum (axis=0): [12 15 18]
Row-wise sum    (axis=1): [ 6 15 24]
Column-wise mean(axis=0): [4. 5. 6.]
Row-wise mean   (axis=1): [2. 5. 8.]


Think of it like a spreadsheet: `axis=0` walks *down* each column, `axis=1` walks *across* each row.

## Conditions using `where`

`np.where(condition)` returns the **indices** where a condition is True. `np.where(condition, if_true, if_false)` works like an if/else applied to every element.

In [68]:
a = np.array([10, 20, 30, 40, 50])
indices = np.where(a > 25)

print(indices)
print(a[indices])

(array([2, 3, 4]),)
[30 40 50]


In [69]:
mask = a < 30
a[mask]

array([10, 20])

In [70]:
b = np.array([1, 2, 3, 4, 5])
result = np.where(b % 2 == 0, 'even', 'odd')
print(result)

['odd' 'even' 'odd' 'even' 'odd']


## 📝 Exercise 2: Sales Data Analysis

Tasks:
1. Create a 4×7 array of random sales between \$100 and \$500.
2. Apply a 10% tax to sales exceeding \$300 (multiply those values by 1.10).
3. Find the average sales for weekends (last 2 columns).
4. Identify days with sales > \$400 (boolean mask).

Try it yourself first.

In [ ]:
# Your code here
s = np.random.uniform(100,500,size=(4,7))

s[s >300]=s[s >300] *1.10



<details>
<summary>💡 Click to see one possible solution</summary>

```python
np.random.seed(2)
sales = np.random.uniform(100, 500, size=(4, 7))

sales[sales > 300] *= 1.10                       # step 2
weekend_avg = sales[:, -2:].mean()                # step 3
high_sales_mask = sales > 400                     # step 4

print("Weekend average sales:", weekend_avg)
print("High sales mask:\n", high_sales_mask)
```
</details>

In [72]:
# Solution
np.random.seed(2)
sales = np.random.uniform(100, 500, size=(4, 7))

sales[sales > 300] *= 1.10
weekend_avg = sales[:, -2:].mean()
high_sales_mask = sales > 400

print("Weekend average sales:", weekend_avg)
print("High sales mask:\n", high_sales_mask)

Weekend average sales: 212.6436597418063
High sales mask:
 [[False False False False False False False]
 [False False False False False False False]
 [False  True  True False  True False False]
 [False False False False False False False]]


## 🔗 Where this goes next

A few of today's tools show up directly in ML code:
- **`np.dot`** → the weighted sum inside linear regression / a neural network neuron
- **Broadcasting** → normalizing features (`(X - X.mean(axis=0)) / X.std(axis=0)`) uses exactly the pattern from today
- **Boolean masking** → filtering rows of training data by a condition
- **`axis`-aware aggregation** → computing per-feature statistics across a whole dataset

## ✅ Cheat sheet

| Task | Function |
|---|---|
| Create | `np.array`, `np.zeros`, `np.ones`, `np.full`, `np.arange`, `np.linspace` |
| Random | `np.random.rand/randint/uniform`, `np.random.seed` |
| Shape | `.reshape`, `.flatten`, `np.resize`, `.shape`, `.ndim`, `.size` |
| Math | `+ - * / **`, `np.sqrt`, `np.dot` |
| Combine | `np.append`, `np.insert`, `np.delete`, `np.concatenate`, `np.vstack`, `np.hstack` |
| Explore | `np.sort`, `np.unique` |
| Aggregate | `np.mean/median/std/sum/min/max`, always check `axis=` |
| Select | slicing `[:]`, fancy indexing `[[...]]`, boolean masks, `np.where` |

**Next session:** Pandas — where these same ideas (indexing, boolean masks, axis, aggregation) get applied to real, labeled, tabular data.